<a href="https://colab.research.google.com/github/kjfcvx12/Colab/blob/main/04_20_p1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conv2d - 3번 (3, 16) - (16,32) - (32,64)

에폭은 10번

훈련, 검증, 평가로 나눠서 작업하기

모델 평가도 한다.

예측정도를 confusion_matrix로 나타내기

In [23]:
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
from torch.utils.data.sampler import SubsetRandomSampler

In [24]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [25]:
composed=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1,),(0.3,))
])

In [26]:
train_data = datasets.CIFAR10(root='./data',
                              train=True,
                              download=True,
                              transform=composed)

test_data = datasets.CIFAR10(root='./data',
                             train=False,
                             download=True,
                             transform=composed)

In [27]:
in_data=list(range(60000)) # [0,1, ..... 59999]
np.random.shuffle(in_data) # 번호표를 무작위로 섞어놓음
split_data=int(np.floor(0.2*60000)) # 12000개는 train-validation용으로 설정 48000개는 train용

# train_index : 12001번째부터 마지막까지(48000개) - 학습용
# valid_index : 처음부터 12000번째까지(12000개) - 검증용
train_index, valid_index=in_data[split_data:], in_data[:split_data]

In [28]:
# DataLoader에 전달될때, 12000, 48000 인덱스에 해당하는 데이터를 무작위로 추출하려고
train_sample=SubsetRandomSampler(train_index)
valid_sample=SubsetRandomSampler(valid_index)

In [29]:
# SubsetRandomSampler - DataLoader
# DataLoader 가 다음 값 요청하면 샘플러가 인덱스 하나를 무작위로 던져주는 구조(호환됨)
train_loader=DataLoader(train_data, batch_size=128, sampler=train_sample)
vaild_loader=DataLoader(train_data, batch_size=128, sampler=valid_sample)
# 60000개를 무작위로 섞은 후 80% 학습 20% 검즘용으로 나눔

# 평가시에 데이터 순서가 섞이면 모델 성능 비교 불안정해짐 -> 섞지 않음
test_loader=DataLoader(test_data, batch_size=128, shuffle=False) # 10000개

In [30]:
class LeNet(nn.Module):
  def __init__(self):
    super().__init__()
    # 입력 -> 특징추출
    self.conv1=nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2)
    self.conv2=nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5)
    self.conv3=nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5)
    # 특징 -> 분류
    self.maxpool=nn.MaxPool2d(2) # 2*2
    self.relu=nn.ReLU()
    # 2번씩 conv, maxpool 시킨 결과
    self.fc1 = nn.Linear(64 * 2 * 2, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    x=self.relu(self.conv1(x))
    x=self.maxpool(x)
    x=self.relu(self.conv2(x))
    x=self.maxpool(x)
    x=self.relu(self.conv3(x))
    x=self.maxpool(x)
    # 2D -> 1D
    # fully connected에 전달할 준비!!
    x=torch.flatten(x,1)
    # FC Lavaer
    x=self.relu(self.fc1(x))
    x=self.relu(self.fc2(x))
    out=self.fc3(x)
    return out